In [ ]:
https://colab.research.google.com/drive/1v3iqmoHxvcihzNlysVmVlmAqkZy8T3tc#scrollTo=EHQp85RffhMW

この内容をローカルで実行したい。
ただし、テストはメタデータフィルターでやりたいので以下を良い感じに流用する。


SAMPLE_TEXTS_WITH_METADATA = [
    {
        "text": (
            "今日は素晴らしい一日でした。朝に近所の公園を散歩し、桜が満開で癒されました。"
        ),
        "metadata": {
            "full_doc_id": "doc-park",
            "chunk_order_index": 0,
            "topic": "outdoor",
            "mood": "relaxed",
            "keywords": ["散歩", "公園"],
        },
    },
    {
        "text": (
            "仕事で取り組んでいたAIプロジェクトが完了し、チーム全員で大きな達成感を味わいました。"
        ),
        "metadata": {
            "full_doc_id": "doc-project",
            "chunk_order_index": 1,
            "topic": "work",
            "mood": "達成",
            "keywords": ["仕事", "プロジェクト"],
        },
    },
    {
        "text": (
            "週末に初めてパスタを一から作り、苦労しながらもコクのあるカルボナーラが完成しました。"
        ),
        "metadata": {
            "full_doc_id": "doc-cooking",
            "chunk_order_index": 2,
            "topic": "cooking",
            "mood": "挑戦",
            "keywords": ["料理", "パスタ"],
        },
    },
    {
        "text": (
            "村上春樹の小説を読み進めながら、深層学習の技術書で理論も学んでいます。"
        ),
        "metadata": {
            "full_doc_id": "doc-reading",
            "chunk_order_index": 3,
            "topic": "reading",
            "mood": "集中",
            "keywords": ["読書", "本"],
        },
    },
    {
        "text": (
            "友人と映画館で『君の名は。』を鑑賞し、感動的なストーリーに胸が熱くなりました。"
        ),
        "metadata": {
            "full_doc_id": "doc-movie",
            "chunk_order_index": 4,
            "topic": "movie",
            "mood": "感動",
            "keywords": ["映画"],
        },
    },
]


async def _sample_insert_texts(rag_instance: _SampleRAG):
    for item in SAMPLE_TEXTS_WITH_METADATA:
        await rag_instance.ainsert(item["text"].strip(), metadata=item["metadata"])


async def _prepare_sample_rag() -> _SampleRAG:
    rag = _SampleRAG()
    await _sample_insert_texts(rag)
    return rag


async def _run_sample_queries(
    rag_instance: _SampleRAG,
    queries: list[str],
    metadata_filters: dict[str, str],
):
    modes = ["naive", "mini", "light"]
    results: dict[str, dict[str, list[str]]] = {}

    for query in queries:
        mode_results: dict[str, list[str]] = {}
        for mode in modes:
            answer = await rag_instance.aquery(
                query,
                param=QueryParam(
                    mode=mode,
                    metadata_filters=metadata_filters,
                    only_need_context=True,
                ),
            )
            mode_results[mode] = answer
        results[query] = mode_results

    return results


def test_sample_rag_filters_by_full_doc_id():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"full_doc_id": "doc-movie"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("映画について教えて", param=param))

    assert len(results) == 1
    assert "映画" in results[0]


def test_sample_rag_filters_by_multiple_conditions():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "cooking", "mood": "挑戦"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("料理で何を作りましたか？", param=param))

    assert len(results) == 1
    assert "カルボナーラ" in results[0]


def test_sample_rag_filters_accept_stringified_numbers():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"chunk_order_index": "2"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("料理で何を作りましたか？", param=param))

    assert len(results) == 1
    assert "パスタ" in results[0]


def test_sample_rag_filters_no_match_returns_empty_list():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "movie", "mood": "集中"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("読んでいる本について教えて", param=param))

    assert results == []


def test_run_sample_queries_applies_filters_to_each_mode():
    rag = asyncio.run(_prepare_sample_rag())
    metadata_filters = {"topic": "movie"}
    queries = [
        "映画について教えて",
        "散歩について詳しく教えて",
    ]

    results = asyncio.run(_run_sample_queries(rag, queries, metadata_filters))


In [33]:
!nvidia-smi

Fri Oct 24 12:49:20 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 528.24       Driver Version: 528.24       CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ... WDDM  | 00000000:01:00.0  On |                  Off |
|  0%   36C    P2    60W / 450W |    681MiB / 24564MiB |      2%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
!uv run python --version

Python 3.12.11


In [ ]:
# インストールしていく
!uv add pip

In [18]:
!uv add python-dotenv json_repair rouge numpy pandas tiktoken nltk pipmaster

Resolved 175 packages in 775ms
Prepared 3 packages in 1.48s
Uninstalled 1 package in 1ms
Installed 3 packages in 10ms
 + ascii-colors==0.11.4
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + pipmaster==1.0.9


In [16]:
!uv add sentence_transformers

Resolved 173 packages in 1.03s
Prepared 13 packages in 25.02s
Uninstalled 1 package in 2ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 16 packages in 8.43s
 + filelock==3.20.0
 + fsspec==2025.9.0
 + huggingface-hub==0.36.0
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + mpmath==1.3.0
 + networkx==3.5
 + pillow==12.0.0
 + safetensors==0.6.2
 + scikit-learn==1.7.2
 + scipy==1.16.2
 + sentence-transformers==5.1.2
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + tokenizers==0.22.1
 + torch==2.9.0
 + transformers==4.57.1


In [52]:
!uv add openai tenacity protobuf sentencepiece pytest

Resolved 184 packages in 1.19s
Prepared 2 packages in 1.40s
Uninstalled 1 package in 1ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 4 packages in 117ms
 + iniconfig==2.3.0
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + pluggy==1.6.0
 + pytest==8.4.2


In [1]:
# GPUを認識させる
# 現在のPyTorchを完全にアンインストール
!uv run python -m pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.9.0
Uninstalling torch-2.9.0:
  Successfully uninstalled torch-2.9.0
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121


Uninstalled 2 packages in 2.53s
Installed 1 package in 10.32s


In [2]:
# CUDA版を再インストール
!uv run python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.1-py3-none-any.whl (6.2 MB)

  Attempting uninstall: sympy

    Found existing installation: sympy 1.14.0

   ---------------------------------------- 0/4 [sympy]
    Uninstalling sympy-1.14.0:
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
      Successfully uninstalled sympy-1.14.0
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
   -------------------

Installed 1 package in 14.83s
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
minirag-hku 0.0.2 requires torch[cuda]>=2.8.0, but you have torch 2.5.1+cu121 which is incompatible.


In [ ]:
!uv lock --upgrade

In [ ]:
!uv sync

In [6]:
!python -c "import sys, torch; print('python exe:', sys.executable); print('python version:', sys.version.splitlines()[0]); print('torch.__version__:', torch.__version__); print('torch.version.cuda:', torch.version.cuda); print('torch.cuda.is_available():', torch.cuda.is_available())"

python exe: C:\Users\kbpsh\OneDrive\development\project\minirag_dayo\.venv\Scripts\python.exe
python version: 3.12.11 (main, Jun 26 2025, 21:17:44) [MSC v.1944 64 bit (AMD64)]
torch.__version__: 2.5.1+cu121
torch.version.cuda: 12.1
torch.cuda.is_available(): True


In [1]:
# GPU を反映させるには一旦Jupyter Notebookサーバーを再起動させる必要あり
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device count:', torch.cuda.device_count())
    print('Current device:', torch.cuda.current_device())
    print('Device name:', torch.cuda.get_device_name(torch.cuda.current_device()))

PyTorch version: 2.5.1+cu121
CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 4090


In [2]:
# feature-metadata-filtering ブランチでOK
!git branch

  codex/add-metadata-filter-to-query
* feature-metadata-filtering
  fix-metadata-filters
  main
  metadata-filter


In [3]:
# 必要なライブラリのインポート
import os
import tempfile
from minirag import MiniRAG, QueryParam
from minirag.llm.hf import (
    hf_model_complete,
    hf_embed,
)
# from minirag.llm.openai import openrouter_openai_complete
from minirag.llm.openai import openai_complete_if_cache
from minirag.utils import EmbeddingFunc
from minirag.utils import (
    wrap_embedding_func_with_attrs,
    locate_json_string_body_from_string,
    safe_unicode_decode,
    logger,
)
from transformers import AutoModel, AutoTokenizer
import asyncio
import warnings
warnings.filterwarnings('ignore')

## 環境変数

In [4]:
import os
from dotenv import load_dotenv

# .envファイルを読み込む
load_dotenv()

# 環境変数を取得する
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')


os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY


print(f"GEMINI_API_KEY: {GEMINI_API_KEY[:5]}********************")
print(f"HF_TOKEN: {HF_TOKEN[:5]}********************")
print(f"OPENROUTER_API_KEY: {OPENROUTER_API_KEY[:5]}********************")

GEMINI_API_KEY: AIzaS********************
HF_TOKEN: hf_UK********************
OPENROUTER_API_KEY: sk-or********************


## セッティング

日本語用埋め込みモデル

In [5]:
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("cl-nagoya/ruri-v3-30m", device=device)

# Ruri v3 employs a 1+3 prefix scheme to distinguish between different types of text inputs:
# "" (empty string) is used for encoding semantic meaning.
# "トピック: " is used for classification, clustering, and encoding topical information.
# "検索クエリ: " is used for queries in retrieval tasks.
# "検索文書: " is used for documents to be retrieved.
sentences = [
    "川べりでサーフボードを持った人たちがいます",
    "サーファーたちが川べりに立っています",
    "トピック: 瑠璃色のサーファー",
    "検索クエリ: 瑠璃色はどんな色？",
    "検索文書: 瑠璃色（るりいろ）は、紫みを帯びた濃い青。名は、半貴石の瑠璃（ラピスラズリ、英: lapis lazuli）による。JIS慣用色名では「こい紫みの青」（略号 dp-pB）と定義している[1][2]。",
]

embeddings = model.encode(sentences, convert_to_tensor=True)
print(embeddings.size())
# [5, 256]

similarities = F.cosine_similarity(embeddings.unsqueeze(0), embeddings.unsqueeze(1), dim=2)
print(similarities)
# [[1.0000, 0.9540, 0.8512, 0.7322, 0.7274],
#  [0.9540, 1.0000, 0.8531, 0.7437, 0.7305],
#  [0.8512, 0.8531, 1.0000, 0.8910, 0.8649],
#  [0.7322, 0.7437, 0.8910, 1.0000, 0.9479],
#  [0.7274, 0.7305, 0.8649, 0.9479, 1.0000]]

torch.Size([5, 256])
tensor([[1.0000, 0.9539, 0.8513, 0.7321, 0.7273],
        [0.9539, 1.0000, 0.8531, 0.7436, 0.7305],
        [0.8513, 0.8531, 1.0000, 0.8908, 0.8647],
        [0.7321, 0.7436, 0.8908, 1.0000, 0.9477],
        [0.7273, 0.7305, 0.8647, 0.9477, 1.0000]], device='cuda:0')


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
# 埋め込みモデルの設定
EMBEDDING_MODEL = "cl-nagoya/ruri-v3-30m"
# LLMの設定
# LLM_MODEL = "Qwen/Qwen3-1.7B"  # または "Qwen/Qwen3-4B", "Qwen/Qwen3-1.7B" など
# LLM_MODEL = "jaeyong2/Qwen2.5-3B-Instruct-Ja-SFT"
# LLM_MODEL = "qwen/qwen3-235b-a22b:free"         # 精度が足りなくてJSONのパースで失敗する
LLM_MODEL = "qwen/qwen3-30b-a3b-instruct-2507"



# 作業ディレクトリの作成
WORKING_DIR = "/tmp/minirag_demo"
os.makedirs(WORKING_DIR, exist_ok=True)

print(f"作業ディレクトリ: {WORKING_DIR}")


# DATA_PATH = args.datapath
# QUERY_PATH = args.querypath
# OUTPUT_PATH = args.outputpath
# print("USING LLM:", LLM_MODEL)
# print("USING WORKING DIR:", WORKING_DIR)

作業ディレクトリ: /tmp/minirag_demo


In [8]:
async def openrouter_openai_complete(
    prompt,
    system_prompt=None,
    history_messages=[],
    keyword_extraction=False,
    api_key: str = None,
    **kwargs,
) -> str:
    # if api_key:
    #     os.environ["OPENROUTER_API_KEY"] = api_key

    keyword_extraction = kwargs.pop("keyword_extraction", None)
    result = await openai_complete_if_cache(
        LLM_MODEL,  # change accordingly
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        **kwargs,
    )
    if keyword_extraction:  # TODO: use JSON API
        return locate_json_string_body_from_string(result)
    return result

In [9]:
%pwd

%ls

 ドライブ C のボリューム ラベルは Windows です
 ボリューム シリアル番号は F444-D319 です

 C:\Users\kbpsh\OneDrive\development\project\minirag_dayo のディレクトリ

2025/10/24  21:23    <DIR>          .
2025/10/23  22:48    <DIR>          ..
2025/10/24  21:23               255 .cursorignore
2025/10/24  12:42               202 .env
2025/10/24  21:23             2,276 .gitignore
2025/10/24  12:55    <DIR>          .ipynb_checkpoints
2025/10/21  08:54               483 .pre-commit-config.yaml
2025/10/24  12:24    <DIR>          .venv
2025/10/21  11:32                 0 ★ディレクトリ名は適当。これはMiniRAGをそのままクローンしたのとほぼ同じ.txt
2025/10/21  08:56    <DIR>          assets
2025/10/21  08:54               234 Communication.md
2025/10/21  08:56    <DIR>          dataset
2025/10/21  08:54               420 docker-compose.yml
2025/10/21  08:54             1,210 Dockerfile
2025/10/21  09:12    <DIR>          docs
2025/10/21  08:56    <DIR>          graph-visuals
2025/10/21  08:54             1,067 LICENSE
2025/10/24  12:22             3,783 main.py


In [10]:
# MiniRAGインスタンスの作成
rag = MiniRAG(
    working_dir=WORKING_DIR,

    # llm_model_func=hf_model_complete,
    llm_model_func=openrouter_openai_complete,

    llm_model_max_token_size=200,
    llm_model_name=LLM_MODEL,
    embedding_func=EmbeddingFunc(
        embedding_dim=256,
        max_token_size=1000,
        func=lambda texts: hf_embed(
            texts,
            tokenizer=AutoTokenizer.from_pretrained(EMBEDDING_MODEL),
            embed_model=AutoModel.from_pretrained(EMBEDDING_MODEL),
        ),
    ),
)

print("MiniRAGが初期化されました！")

INFO:nano-vectordb:Load (189, 256) data
INFO:nano-vectordb:Init {'embedding_dim': 256, 'metric': 'cosine', 'storage_file': '/tmp/minirag_demo\\vdb_entities.json'} 189 data
INFO:nano-vectordb:Load (189, 256) data
INFO:nano-vectordb:Init {'embedding_dim': 256, 'metric': 'cosine', 'storage_file': '/tmp/minirag_demo\\vdb_entities_name.json'} 189 data
INFO:nano-vectordb:Load (486, 256) data
INFO:nano-vectordb:Init {'embedding_dim': 256, 'metric': 'cosine', 'storage_file': '/tmp/minirag_demo\\vdb_relationships.json'} 486 data
INFO:nano-vectordb:Load (5, 256) data
INFO:nano-vectordb:Init {'embedding_dim': 256, 'metric': 'cosine', 'storage_file': '/tmp/minirag_demo\\vdb_chunks.json'} 5 data
INFO:minirag:Loaded document status storage with 8 records


MiniRAGが初期化されました！


In [11]:
# これをやっておかないと HuggingFace の認証で失敗する

from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, token=HF_TOKEN)
model = AutoModelForMaskedLM.from_pretrained(EMBEDDING_MODEL, token=HF_TOKEN)

Some weights of ModernBertForMaskedLM were not initialized from the model checkpoint at cl-nagoya/ruri-v3-30m and are newly initialized: ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# prompt: ロードしたmodelの開放

# メモリ解放のためにモデルを削除
del model
del tokenizer

# PyTorchのキャッシュをクリア (GPUを使用している場合)
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Pythonのガベージコレクションを実行
import gc
gc.collect()

print("モデルとトークナイザーが解放されました。")

モデルとトークナイザーが解放されました。


In [78]:
# 約12分かかった
import time
start_time = time.time()

# サンプルテキストデータ
sample_texts = [
    """
今日は素晴らしい一日でした。朝早く起きて、近所の公園を散歩しました。
桜の花が満開で、とても美しかったです。午後は友人と映画を見に行きました。
「君の名は。」という映画で、とても感動的でした。
夜は家族と一緒に夕食を取り、楽しい時間を過ごしました。
""",
    """
昨日は仕事で大きなプロジェクトが完了しました。
チーム全員で3ヶ月間取り組んできたAIシステムの開発が終わりました。
機械学習モデルの精度が95%を超え、クライアントからも高い評価をいただきました。
今夜はチームメンバーと祝賀会を開く予定です。
""",
    """
週末は料理に挑戦しました。初めてパスタを一から作ってみました。
小麦粉から麺を作るのは思っていたより難しかったですが、
最終的にはとても美味しいカルボナーラができました。
次回はリゾットに挑戦してみたいと思います。
""",
    """
読書が趣味で、最近は村上春樹の「ノルウェイの森」を読んでいます。
主人公の心情描写がとても繊細で、引き込まれます。
また、技術書も読んでおり、「深層学習」について学んでいます。
理論と実践のバランスが取れた良い本だと思います。
"""
]

# データの挿入
print("データを挿入中...")

async def insert_texts(rag_instance, texts):
    for i, text in enumerate(texts):
        print(f"テキスト {i+1}/{len(texts)} を挿入中...")
        await rag_instance.ainsert(text.strip())

    print("\nすべてのデータが挿入されました！")


# イベントループが既に実行中の場合
try:
    await insert_texts(rag, sample_texts)
except RuntimeError:
    # 新しいループで実行
    asyncio.run(insert_texts(rag, sample_texts))

end_time = time.time()
elapsed_time = end_time - start_time
print(f"処理時間: {elapsed_time:.4f}秒")

INFO:minirag:No new unique documents were found.
INFO:minirag:No documents to process
INFO:minirag:Performing entity extraction on newly processed chunks


データを挿入中...
テキスト 1/4 を挿入中...


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/c

⠙ Processed 1 chunks, 9 entities(duplicated), 12 relations(duplicated)

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


⠇ Processed 8 chunks, 89 entities(duplicated), 103 relations(duplicated)

INFO:minirag:Inserting 60 vectors to entities


Generating embeddings: 100%|███████████████| 2/2 [00:03<00:00,  1.94s/batch]
INFO:minirag:Inserting 60 vectors to entities
Generating embeddings: 100%|███████████████| 2/2 [00:03<00:00,  1.79s/batch]
INFO:minirag:Inserting 60 vectors to entities_name
Generating embeddings: 100%|███████████████| 2/2 [00:02<00:00,  1.36s/batch]
INFO:minirag:Inserting 83 vectors to relationships
Generating embeddings: 100%|███████████████| 3/3 [00:05<00:00,  1.67s/batch]
INFO:minirag:Writing graph with 143 nodes, 316 edges



すべてのデータが挿入されました！
処理時間: 244.5082秒


In [79]:
# 約3分かかった

# # サンプルクエリ
# queries = [
#     "映画について教えて",
#     "仕事のプロジェクトはどうでしたか？",
#     "料理で何を作りましたか？",
#     "読んでいる本について教えて",
#     "散歩について詳しく教えて"
# ]

# # 各モードでクエリを実行。この3つがある
# modes = ["naive", "mini", "light"]

# for query in queries:
#     print(f"\n{'='*50}")
#     print(f"クエリ: {query}")
#     print(f"{'='*50}")

#     for mode in modes:
#         print(f"\n--- {mode.upper()}モード ---")
#         try:
#             answer = rag.query(query, param=QueryParam(mode=mode))     # .replace("\n", "").replace("\r", "")
#             print(f"回答: {answer}")
#         except Exception as e:
#             print(f"エラー: {e}")


async def run_queries(rag_instance, queries):
    # 各モードでクエリを実行。この3つがある
    modes = ["naive", "mini", "light"]

    for query in queries:
        print(f"\n{'='*50}")
        print(f"クエリ: {query}")
        print(f"{'='*50}")

        for mode in modes:
            print(f"\n--- {mode.upper()}モード ---")
            try:
                # 非同期でクエリを実行
                answer = await rag_instance.aquery(query, param=QueryParam(mode=mode))
                print(f"回答: {answer}")
            except Exception as e:
                print(f"エラー: {e}")


start_time = time.time()

sample_queries = [
    "映画について教えて",
    "仕事のプロジェクトはどうでしたか？",
    "料理で何を作りましたか？",
    "読んでいる本について教えて",
    "散歩について詳しく教えて"
]

# イベントループが既に実行中の場合
try:
    await run_queries(rag, sample_queries)
except RuntimeError:
    # 新しいループで実行
    asyncio.run(run_queries(rag, sample_queries))


end_time = time.time()
elapsed_time = end_time - start_time
print(f"処理時間: {elapsed_time:.4f}秒")


クエリ: 映画について教えて

--- NAIVEモード ---


INFO:minirag:Query: 映画について教えて, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Truncate 5 to 5 chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


回答: 申し訳ありませんが、提供された文書の中に映画に関する情報は含まれていません。したがって、映画についての回答をすることはできません。

--- MINIモード ---


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:minirag:Query: 映画, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Query: 映画について教えて, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Query: 映画について教えて, top_k: 30, cosine_better_than_threshold: 0.2
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


回答: 映画は、主人公と友人が午後に共同で体験した重要な文化活動であり、特に『君の名は。』という作品を通じて感情的な共鳴や深い感動を得る機会となった。この映画鑑賞は単なる娯楽に留まらず、二人の関係を深める共同体験ともなった。映画は午後に観られたため、日常の中でも特別な時間として位置づけられており、物語の展開と登場人物の心情に強く惹き込まれたとされている。『君の名は。』は、孤独や運命、記憶のテーマを通じて観客に強い感情的インパクトを与えることで知られており、この体験もその一例である。

映画鑑賞は、観る側が感情を共感し、内面的な反応を生む重要な行為である。本件では、映画を通じて「感動的な体験」が共有され、その影響は観た後も長く残ったとされる。また、映画というメディアは社会的つながりを促進する場としても機能しており、友人同士の交流の中での共有体験としての価値が強調されている。文化的なイベントとしての側面も持ち、単なる娯楽を超えて、人間関係や自己理解の深化に寄与する可能性を持つ。

これらの特徴から、映画鑑賞は単なる時間の使い方ではなく、心理的・社会的・文化的に豊かな意味を持つ人間の体験の一部であると言える。『君の名は。』のような作品は、具体的なストーリーの魅力だけでなく、共感の核を提示することで、観る人をその世界に引き込む力を持っている。現代社会においても、映画は人々が自分と他者、そして時間との関係を見直す貴重な手段として、その役割を果たしている。

--- LIGHTモード ---


...

回答: 申し訳ありませんが、提供された文書には散歩に関する情報が含まれていません。したがって、散歩について詳しく説明することはできません。

--- MINIモード ---


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:minirag:Query: 散歩, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Query: 散歩について詳しく教えて, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Query: 散歩について詳しく教えて, top_k: 30, cosine_better_than_threshold: 0.2
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


回答: ### 散歩の意味と背景

「散歩」は、朝に近所の公園で行われる日常的なリラクゼーション活動として、特に自然との触れ合いを目的とした行動として描写されています。この行為は単なる移動ではなく、心を落ち着かせ、一日の良いスタートを切りたいという意図を含んでおり、主人公の精神状態や生活の質を反映する象徴的な行動でもあります。複数の記述から見ると、散歩は自然を鑑賞するための時間であり、朝の光や満開の桜といった季節の美しさに目を向ける体験が含まれている。そのような環境の中での散歩は、単なる運動以上の意味を持ち、心の安らぎや自己との対話の場として機能している。

---

### 散歩の文化的・心理的意義

散歩は日本文化においても古くから重んじられてきた行動であり、特に朝の散歩は「一日のリズム」を整えるための習慣とされる。この活動は、心身のリフレッシュや冥想に近い効果をもたらすとされ、自然に触れることでストレスが軽減され、集中力や創造力が高まるという効果が知られている。今回の記述では、主人公が朝の公園を歩くことで、自然と向き合い、その中で「良い気分」や「心の基盤」を作り上げていることが強調されています。これは、単に外を歩くという物理的行為ではなく、内面的な発展や自己探求のプロセスを含んでいることを示しており、散歩が「心の生活」を豊かにする手段としての役割を果たしている。

---

### 散歩と他のライフスタイル行動の関係

散歩と他の日常的習慣（例：読書、料理など）との関係も興味深い。例えば、「読書が趣味」という人物は、『ノルウェイの森』という文学作品と『深層学習』という技術書を読んでいるとされ、精神的啓発と学問的成長を両立させている。このように、散歩も同様に「内省」や「自己成長」の場として機能しており、読書と並んで、個人の内面世界を育てる重要な活動の一つであると言える。さらに、料理に挑戦するような日常的な実践活動と組み合わせることで、散歩は「身体」と「心」の両面でのバランスを取る生活スタイルの一部として位置づけられる。

---

### 結論

「散歩」は単なる移動手段を越えた、精神的な安らぎと自然との調和を求める行為であり、朝の時間帯に近所の公園を歩くこの行動には明確な儀式的な側面がある。自然の美しさに触れながら心を整える、その行為は現代人の忙しい日常の中で

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:minirag:Query: 歩行, 公園, 散歩道, 心臓病予防, ストレス解消, 歩数, ウォーキング, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Local query uses 60 entites, 36 relations, 5 text units
INFO:minirag:Query: 散歩, 健康, 生活習慣, リラクゼーション, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Global query uses 10 entites, 24 relations, 2 text units
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


回答: 散歩は、主人公の一日の開始を象徴する重要な行動であり、朝の時間帯に近所の公園で行われた。この活動は単なる運動や移動にとどまらず、自然と触れ合い、心を落ち着けるリラクゼーションの時間として機能している。散歩の舞台となった「近所の公園」は、朝早く訪れると、桜の花が満開となっており、その美しさが文章全体の静けさと豊かな感情を形成する重要な要素として描かれている。特に「朝」という時間帯に桜の花が満開であるという時間的・空間的合致は、自然の美しさを深く鑑賞する機会を提供しており、主人公の一日の良い始まりを豊かにしている。

さらに、この散歩は一日の中で「素晴らしい一日」を構成する基盤となる体験である。朝の散歩というリラクゼーションは、一日の流れに沿って自然と一体化する感覚をもたらし、その後の活動——映画鑑賞や家族との夕食——に対する心の準備や余裕を生み出す。文脈上、散歩は単なる行動というだけでなく、心の調子を整え、日々の生活に豊かさを与える「日常の儀礼」として位置づけられている。自然との対話を通じて、心がリセットされ、前向きな気持ちで一日を過ごすための重要な前段階として機能している。

また、この散歩の体験は、時間的にも空間的にも明確な関係を持つ。朝という時間帯と近所の公園という場所が、直接的に結びついており、散歩はこの時間と場所との相乗効果によって成立している。公園には「桜の花」という自然の景観が存在し、その美しさこそがこの散歩の主な見どころとなっている。この景色によって、主人公は心の安らぎを得るとともに、自然の儚さや美しさに対する感動を感じ取っている。こうした体験は、単なる散歩以上の意義を持ち、自己の内面と自然との調和を実感する貴重な瞬間とされている。
処理時間: 239.5034秒


In [13]:
SAMPLE_TEXTS_WITH_METADATA = [
    {
        "text": (
            "今日は素晴らしい一日でした。朝早く起きて、近所の公園を散歩しました。桜の花が満開で、とても美しかったです。午後は友人と映画を見に行きました。「君の名は。」という映画で、とても感動的でした。夜は家族と一緒に夕食を取り、楽しい時間を過ごしました。"
        ),
        "metadata": {
            "full_doc_id": "doc-park",
            "chunk_order_index": 0,
            "topic": "outdoor",
            "mood": "relaxed",
            "keywords": ["散歩", "公園"],
        },
    },
    {
        "text": (
            "昨日は仕事で大きなプロジェクトが完了しました。チーム全員で3ヶ月間取り組んできたAIシステムの開発が終わりました。機械学習モデルの精度が95%を超え、クライアントからも高い評価をいただきました。今夜はチームメンバーと祝賀会を開く予定です。"
        ),
        "metadata": {
            "full_doc_id": "doc-project",
            "chunk_order_index": 1,
            "topic": "work",
            "mood": "達成",
            "keywords": ["仕事", "プロジェクト"],
        },
    },
    {
        "text": (
            "週末は料理に挑戦しました。初めてパスタを一から作ってみました。小麦粉から麺を作るのは思っていたより難しかったですが、最終的にはとても美味しいカルボナーラができました。次回はリゾットに挑戦してみたいと思います。"
        ),
        "metadata": {
            "full_doc_id": "doc-cooking",
            "chunk_order_index": 2,
            "topic": "cooking",
            "mood": "挑戦",
            "keywords": ["料理", "パスタ"],
        },
    },
    {
        "text": (
            "読書が趣味で、最近は村上春樹の「ノルウェイの森」を読んでいます。主人公の心情描写がとても繊細で、引き込まれます。また、技術書も読んでおり、「深層学習」について学んでいます。理論と実践のバランスが取れた良い本だと思います。"
        ),
        "metadata": {
            "full_doc_id": "doc-reading",
            "chunk_order_index": 3,
            "topic": "reading",
            "mood": "集中",
            "keywords": ["読書", "本"],
        },
    },
]


async def _sample_insert_texts(rag_instance):
    for item in SAMPLE_TEXTS_WITH_METADATA:
        # metadatas: dict | list[dict] | None = None
        await rag_instance.ainsert(item["text"].strip(), metadatas=item["metadata"])


async def _prepare_sample_rag():
    await _sample_insert_texts(rag)
    return rag


# メタデータ入りのデータ挿入
await _prepare_sample_rag()

INFO:minirag:No new unique documents were found.
INFO:minirag:No documents to process
INFO:minirag:Performing entity extraction on newly processed chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


⠇ Processed 8 chunks, 87 entities(duplicated), 87 relations(duplicated)

INFO:minirag:Inserting 57 vectors to entities


Generating embeddings:   0%|      | 0/2 [00:00<?, ?batch/s]We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
Generating embeddings: 100%|█| 2/2 [00:08<00:00,  4.32s/bat
INFO:minirag:Inserting 57 vectors to entities
Generating embeddings: 100%|█| 2/2 [00:07<00:00,  3.95s/bat
INFO:minirag:Inserting 57 vectors to entities_name
Generating embeddings: 100%|█| 2/2 [00:02<00:00,  1.38s/bat
INFO:minirag:Inserting 74 vectors to relationships
Generating embeddings: 100%|█| 3/3 [00:10<00:00,  3.44s/bat
INFO:minirag:Writing graph with 206 nodes, 512 edges
INFO:minirag:No new unique documents were found.
INFO:minirag:No documents to process
INFO:minirag:Performing entity extraction on newly processed chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai

⠙ Processed 1 chunks, 5 entities(duplicated), 4 relations(duplicated)

...

Generating embeddings: 100%|█| 2/2 [00:10<00:00,  5.26s/bat
INFO:minirag:Inserting 61 vectors to entities
Generating embeddings: 100%|█| 2/2 [00:10<00:00,  5.07s/bat
INFO:minirag:Inserting 61 vectors to entities_name
Generating embeddings: 100%|█| 2/2 [00:02<00:00,  1.15s/bat
INFO:minirag:Inserting 85 vectors to relationships
Generating embeddings: 100%|█| 3/3 [00:09<00:00,  3.27s/bat
INFO:minirag:Writing graph with 219 nodes, 542 edges
INFO:minirag:No new unique documents were found.
INFO:minirag:No documents to process
INFO:minirag:Performing entity extraction on newly processed chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST

⠇ Processed 8 chunks, 91 entities(duplicated), 105 relations(duplicated)

INFO:minirag:Inserting 62 vectors to entities


Generating embeddings: 100%|█| 2/2 [00:11<00:00,  5.87s/bat
INFO:minirag:Inserting 62 vectors to entities
Generating embeddings: 100%|█| 2/2 [00:11<00:00,  5.54s/bat
INFO:minirag:Inserting 62 vectors to entities_name
Generating embeddings: 100%|█| 2/2 [00:02<00:00,  1.22s/bat
INFO:minirag:Inserting 88 vectors to relationships
Generating embeddings: 100%|█| 3/3 [00:10<00:00,  3.62s/bat
INFO:minirag:Writing graph with 228 nodes, 561 edges


MiniRAG(working_dir='/tmp/minirag_demo', kv_storage='JsonKVStorage', vector_storage='NanoVectorDBStorage', graph_storage='NetworkXStorage', log_level=0, chunk_token_size=1200, chunk_overlap_token_size=100, tiktoken_model_name='gpt-4o-mini', entity_extract_max_gleaning=1, entity_summary_to_max_tokens=500, node_embedding_algorithm='node2vec', node2vec_params={'dimensions': 1536, 'num_walks': 10, 'walk_length': 40, 'window_size': 2, 'iterations': 3, 'random_seed': 3}, embedding_func=<function limit_async_func_call.<locals>.final_decro.<locals>.wait_func at 0x0000027B4A96E8E0>, embedding_batch_num=32, embedding_func_max_async=16, llm_model_func=<function limit_async_func_call.<locals>.final_decro.<locals>.wait_func at 0x0000027BAD720F40>, llm_model_name='qwen/qwen3-30b-a3b-instruct-2507', llm_model_max_token_size=200, llm_model_max_async=16, llm_model_kwargs={}, vector_db_storage_cls_kwargs={}, enable_llm_cache=True, addon_params={}, convert_response_to_json_func=<function convert_response

In [14]:
import pytest

@pytest.mark.asyncio
async def test1():
    param = QueryParam(
        mode="naive",
        only_need_context=False,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]

await test1()

INFO:minirag:Query: 料理で何を作りましたか？, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Truncate 5 to 5 chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


results = 週末に初めてパスタを一から作りました。小麦粉から麺を作る過程は思ったよりも難しく、レシピ通りに進めるだけでも集中力が必要でしたが、最後に完成したカルボナーラはとても美味しかったです。アンチパスタの仕上がりに満足しながらも、生地の扱い方や茹でるタイミングの工夫が次回の課題だと感じました。これからはリゾットに挑戦してみようと思っています。


In [82]:
@pytest.mark.asyncio
async def test2():
    param = QueryParam(
        mode="naive",
        only_need_context=True,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]

await test2()

INFO:minirag:Query: 料理で何を作りましたか？, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Truncate 5 to 5 chunks


results = 週末は料理に挑戦しました。初めてパスタを一から作ってみました。小麦粉から麺を作るのは思っていたより難しかったですが、最終的にはとても美味しいカルボナーラができました。次回はリゾットに挑戦してみたいと思います。--New Chunk--
昨日は仕事で大きなプロジェクトが完了しました。チーム全員で3ヶ月間取り組んできたAIシステムの開発が終わりました。機械学習モデルの精度が95%を超え、クライアントからも高い評価をいただきました。今夜はチームメンバーと祝賀会を開く予定です。--New Chunk--
昨日は仕事で大きなプロジェクトが完了しました。
チーム全員で3ヶ月間取り組んできたAIシステムの開発が終わりました。
機械学習モデルの精度が95%を超え、クライアントからも高い評価をいただきました。
今夜はチームメンバーと祝賀会を開く予定です。--New Chunk--
読書が趣味で、最近は村上春樹の「ノルウェイの森」を読んでいます。主人公の心情描写がとても繊細で、引き込まれます。また、技術書も読んでおり、「深層学習」について学んでいます。理論と実践のバランスが取れた良い本だと思います。--New Chunk--
読書が趣味で、最近は村上春樹の「ノルウェイの森」を読んでいます。
主人公の心情描写がとても繊細で、引き込まれます。
また、技術書も読んでおり、「深層学習」について学んでいます。
理論と実践のバランスが取れた良い本だと思います。


In [15]:
@pytest.mark.asyncio
async def test3():
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "cooking", "mood": "挑戦"},
        only_need_context=False,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]

await test3()

INFO:minirag:Query: 料理で何を作りましたか？, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Truncate 1 to 1 chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


results = 週末に料理に挑戦し、初めてパスタを一から作ってみました。この経験では、小麦粉から手作りの麺を捏ねることから始まり、実際に手で作る過程が想像以上に難しかったと感じました。しかし、粘り気のある生地を十分にこね、延ばして細く切る作業を経て、完成した自家製パスタは見た目も満足のいくものになりました。

そのパスタを使ったのは、伝統的なイタリア料理であるカルボナーラです。卵黄とチーズ、ベーコン、スパイスを用いてソースを調製し、熱々のパスタに絡めることで、クリーミーで濃厚な風味のカルボナーラが完成しました。実際に口に含んだ瞬間、想像以上に美味しかったと感じ、これまで購入した冷凍パスタとは比べ物にならない満足感を得られました。

この経験を機に、次回はリゾットへの挑戦を計画しているとのこと。リゾットは米の粒がやわらかく、ミルキーな食感が特徴で、トウモロコシやキノコ、チーズなどさまざまな具材との相性も良いと言われています。手作りパスタの成功を糧に、リゾットも自分の手で丁寧に仕上げることで、さらに料理の腕が上がることを期待したいです。


In [84]:
@pytest.mark.asyncio
async def test4():
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "cooking", "mood": "集中"},    # 集中 にしたからゼロ件になった
        only_need_context=False,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]

await test4()

INFO:minirag:Query: 料理で何を作りましたか？, top_k: 60, cosine_better_than_threshold: 0.2


results = 


In [85]:
@pytest.mark.asyncio
async def test5():
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "cooking"},
        only_need_context=False,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]

await test5()

INFO:minirag:Query: 料理で何を作りましたか？, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Truncate 1 to 1 chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


results = 週末に料理に挑戦し、初めてパスタを一から作ってみました。特に小麦粉から麺を手作りするのは想像以上に難しかったですが、粘り気のある生地をうまく扱い、最後には美しいストレートのパスタに仕上げることができました。完成したパスタは、特製のクリームソースとベーコン、パルメジアーノ・レッジャーノを合わせたカルボナーラに仕上げ、香り高く、コクのある味わいに仕上がりました。初めての手作りパスタということもあり、工程のすべてに集中しましたが、その分、完成した際の達成感と美味しい結果にとても満足しています。

次回の料理チャレンジとしては、リゾットに挑戦してみたいと考えています。パスタとは違い、グラブリッチェというルールに従い、ゆっくりとスープを加えながら混ぜていくリゾットは、火加減や水分の調整が重要だと言われており、まさに「味の芸術」ともいえる料理です。これまでのパスタづくりで得た経験を活かし、ぜひリゾットの奥深さを味わいたいと思っています。


In [18]:
@pytest.mark.asyncio
async def test6():
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "work"},    # これだと料理はヒットしない
        only_need_context=False,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]

await test6()

INFO:minirag:Query: 料理で何を作りましたか？, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Truncate 1 to 1 chunks
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


results = 申し訳ありませんが、提供された文書には料理についての情報が含まれていません。文書では、AIシステム開発の完了やチームでの祝賀会の予定について述べられていますが、具体的に何の料理を作ったかについては記載されていません。


In [19]:
@pytest.mark.asyncio
async def test7():
    param = QueryParam(
        mode="mini",
        metadata_filters={"topic": "work"},    # mini だとグラフをたどってヒットしちゃう？？ → 仕様通りじゃない
        only_need_context=False,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]

await test7()

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:minirag:Query: 料理, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Query: 何を作りました, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Query: 料理で何を作りましたか？, top_k: 120, cosine_better_than_threshold: 0.2
INFO:minirag:Query: 料理で何を作りましたか？, top_k: 30, cosine_better_than_threshold: 0.2
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


results = 申し訳ありませんが、提供されたデータには料理に関する情報が含まれていません。したがって、何の料理を作ったかを回答することはできません。


In [20]:
@pytest.mark.asyncio
async def test8():
    param = QueryParam(
        mode="light",
        metadata_filters={"topic": "work"},    # light モードもヒットしてしまうので仕様通りじゃない
        only_need_context=False,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]

await test8()

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:minirag:Query: 料理の種類, 材料, 調理法, 味付け, レシピ, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Local query uses 1 entites, 10 relations, 1 text units
INFO:minirag:Query: 料理, 作成, 調理, 食事, top_k: 60, cosine_better_than_threshold: 0.2
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


results = 申し訳ありませんが、提供されたデータには料理や作られた料理に関する情報が含まれていないため、お答えすることはできません。


In [ ]:
Jules で実装してもらったやつだけど、駄目そう。

In [ ]:
lightとminiではmetadata_filtersが機能しないケースがあります。調査して修正してください。
→Codexの修正を反映したらできた？？ 簡単なテストは問題ない。

In [75]:
import pytest

async def _run_sample_queries(
    rag_instance,
    queries: list[str],
    metadata_filters: dict[str, str],
):
    modes = ["naive", "mini", "light"]
    results: dict[str, dict[str, list[str]]] = {}

    for query in queries:
        mode_results: dict[str, list[str]] = {}
        for mode in modes:
            print("============ 現在の検索モード ===============")
            print(mode)
            answer = await rag_instance.aquery(
                query,
                param=QueryParam(
                    mode=mode,
                    metadata_filters=metadata_filters,
                    only_need_context=True,
                ),
            )
            mode_results[mode] = answer
        results[query] = mode_results

    return results


@pytest.mark.asyncio
async def test_sample_rag_filters_by_full_doc_id():
    param = QueryParam(
        mode="naive",
        metadata_filters={"full_doc_id": "doc-movie"},
        only_need_context=True,
    )
    results = await rag.aquery("映画について教えて", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "映画" in results[0]


@pytest.mark.asyncio
async def test_sample_rag_filters_by_multiple_conditions():
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "cooking", "mood": "挑戦"},
        only_need_context=True,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")
    
    # assert len(results) == 1
    # assert "カルボナーラ" in results[0]


@pytest.mark.asyncio
async def test_sample_rag_filters_accept_stringified_numbers():
    param = QueryParam(
        mode="naive",
        metadata_filters={"chunk_order_index": "2"},
        only_need_context=True,
    )
    results = await rag.aquery("料理で何を作りましたか？", param=param)
    print(f"results = {results}")

    # assert len(results) == 1
    # assert "パスタ" in results[0]


@pytest.mark.asyncio
async def test_sample_rag_filters_no_match_returns_empty_list():
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "movie", "mood": "集中"},
        only_need_context=True,
    )
    results = await rag.aquery("読んでいる本について教えて", param=param)
    print(f"results = {results}")

    # assert results == []


@pytest.mark.asyncio
async def test_run_sample_queries_applies_filters_to_each_mode():
    metadata_filters = {"topic": "movie"}
    queries = [
        "映画について教えて",
        "散歩について詳しく教えて",
    ]
    results = await _run_sample_queries(rag, queries, metadata_filters)
    print(results)


In [64]:
await test_run_sample_queries_applies_filters_to_each_mode()

============ 現在の検索モード ===============
naive


INFO:minirag:Query: 映画について教えて, top_k: 60, cosine_better_than_threshold: 0.2


============ 現在の検索モード ===============
mini


Exception ignored in: <coroutine object _run_sample_queries at 0x0000011A49B0BB00>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object _prepare_sample_rag at 0x00000119BEE29C00>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object MiniRAG.aquery at 0x0000011A49C05F20>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'


CancelledError: 

In [72]:
await test_sample_rag_filters_no_match_returns_empty_list()

INFO:minirag:Query: 読んでいる本について教えて, top_k: 60, cosine_better_than_threshold: 0.2


results = 


In [73]:
await test_sample_rag_filters_accept_stringified_numbers()

INFO:minirag:Query: 料理で何を作りましたか？, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Truncate 1 to 1 chunks


results = 週末は料理に挑戦しました。初めてパスタを一から作ってみました。小麦粉から麺を作るのは思っていたより難しかったですが、最終的にはとても美味しいカルボナーラができました。次回はリゾットに挑戦してみたいと思います。


In [76]:
await test_sample_rag_filters_by_multiple_conditions()

INFO:minirag:Query: 料理で何を作りましたか？, top_k: 60, cosine_better_than_threshold: 0.2
INFO:minirag:Truncate 1 to 1 chunks


results = 週末は料理に挑戦しました。初めてパスタを一から作ってみました。小麦粉から麺を作るのは思っていたより難しかったですが、最終的にはとても美味しいカルボナーラができました。次回はリゾットに挑戦してみたいと思います。


In [77]:
await test_sample_rag_filters_by_full_doc_id()

INFO:minirag:Query: 映画について教えて, top_k: 60, cosine_better_than_threshold: 0.2


results = 
